In [1]:
"""
Full AC solver for the Andrews-Curtis Conjecture training set.
Uses the ac-r2-v1 move spec.
Run from: competition/examples/
"""

import json
from collections import deque
from pathlib import Path

# ============================================================
# STEP 1: LOCATE FILES
# ============================================================
EXAMPLES_DIR = Path(".")
AC_JSONL = EXAMPLES_DIR / "ac.jsonl"
STABLE_JSONL = EXAMPLES_DIR / "stable_ac.jsonl"

print(f"AC file:     {AC_JSONL.resolve()}")
print(f"Stable file: {STABLE_JSONL.resolve()}")
assert AC_JSONL.exists(), "ac.jsonl not found. Run this from competition/examples/"

# ============================================================
# STEP 2: PARSER
# ============================================================
def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def decode(word):
    return (word.replace("X", "x^-1")
                .replace("Y", "y^-1"))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    generators = gens_part.strip().split()
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        rel = chunk.replace("= 1", "").strip()
        relators.append(encode(rel))
    return generators, relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            gens, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "generators": gens,
                "relators": rels,
            })
    return problems

# ============================================================
# STEP 3: FREE REDUCTION AND INVERSION
# ============================================================
def is_inv(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    stack = []
    for ch in word:
        if stack and is_inv(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    out = []
    for ch in reversed(word):
        out.append(ch.upper() if ch.islower() else ch.lower())
    return "".join(out)

# ============================================================
# STEP 4: APPLY ONE MOVE (ac-r2-v1 spec)
# ============================================================
def apply_move(rels, m):
    r = list(rels)
    if   m == 0:  r[0] = free_reduce("x" + r[0] + "X")
    elif m == 1:  r[0] = free_reduce("X" + r[0] + "x")
    elif m == 2:  r[0] = free_reduce("y" + r[0] + "Y")
    elif m == 3:  r[0] = free_reduce("Y" + r[0] + "y")
    elif m == 4:  r[0] = free_reduce(r[0] + r[1])
    elif m == 5:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif m == 6:  r[0] = free_reduce(invert_word(r[0]))
    elif m == 7:  r[1] = free_reduce("x" + r[1] + "X")
    elif m == 8:  r[1] = free_reduce("X" + r[1] + "x")
    elif m == 9:  r[1] = free_reduce("y" + r[1] + "Y")
    elif m == 10: r[1] = free_reduce("Y" + r[1] + "y")
    elif m == 11: r[1] = free_reduce(r[1] + r[0])
    elif m == 12: r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif m == 13: r[1] = free_reduce(invert_word(r[1]))
    return r

# ============================================================
# STEP 5: STATE KEY FOR VISITED SET
# ============================================================
def state_key(rels):
    return tuple(sorted(rels))

# ============================================================
# STEP 6: BIDIRECTIONAL BFS
# ============================================================
GOAL = ["x", "y"]

def bidirectional_solve(start_rels, max_depth=12):
    start_key = state_key(start_rels)
    goal_key = state_key(GOAL)
    if start_key == goal_key:
        return []

    fwd = deque([(list(start_rels), [])])
    bwd = deque([(list(GOAL), [])])
    fwd_seen = {start_key: []}
    bwd_seen = {goal_key: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in bwd_seen:
                    return path + [m] + bwd_seen[nk]
                if nk not in fwd_seen:
                    fwd_seen[nk] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in fwd_seen:
                    return fwd_seen[nk] + [m] + path
                if nk not in bwd_seen:
                    bwd_seen[nk] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ============================================================
# STEP 7: SANITY CHECK AGAINST KNOWN SOLUTION
# ============================================================
print("\n=== Sanity check: ms-train-0001 known solution ===")
start = ["XyxyY", "xy"]
known = [0, 9, 1, 2, 9, 0, 4, 1]
state = list(start)
for m in known:
    state = apply_move(state, m)
print(f"start: {start}")
print(f"after known moves {known}: {state}")
print(f"goal:  {GOAL}")
print(f"MATCH: {sorted(state) == sorted(GOAL)}")

# ============================================================
# STEP 8: SOLVE FIRST 10 TRAINING PROBLEMS AND WRITE SUBMISSION
# ============================================================
print("\n=== Solving first 10 training problems ===")
problems = load_jsonl(AC_JSONL)[:10]
results = {}

for prob in problems:
    cid = prob["challenge_id"]
    rels = prob["relators"]
    moves = bidirectional_solve(rels, max_depth=12)
    if moves is not None:
        print(f"{cid}: SOLVED in {len(moves)} moves -> {moves}")
        results[cid] = moves
    else:
        print(f"{cid}: NOT SOLVED within depth 12")

with open("practice_submission_3.txt", "w") as f:
    for cid, moves in results.items():
        moves_str = ", ".join(str(m) for m in moves)
        f.write(f"{cid}: [{moves_str}]\n")

print(f"\nWrote {len(results)} solutions to practice_submission_3.txt")

AC file:     C:\Users\Goodluck\AC_Conjecture\Andrews-Curtis\competition\examples\ac.jsonl
Stable file: C:\Users\Goodluck\AC_Conjecture\Andrews-Curtis\competition\examples\stable_ac.jsonl

=== Sanity check: ms-train-0001 known solution ===
start: ['XyxyY', 'xy']
after known moves [0, 9, 1, 2, 9, 0, 4, 1]: ['yXyxYXyyxYx', 'yyxY']
goal:  ['x', 'y']
MATCH: False

=== Solving first 10 training problems ===
ms-train-0001: SOLVED in 5 moves -> [0, 4, 3, 7, 11]
ms-train-0002: SOLVED in 6 moves -> [0, 4, 3, 2, 7, 11]
ms-train-0003: SOLVED in 7 moves -> [0, 4, 3, 3, 7, 11, 2]
ms-train-0004: SOLVED in 8 moves -> [0, 4, 3, 3, 2, 7, 11, 2]
ms-train-0005: SOLVED in 9 moves -> [0, 4, 3, 3, 3, 7, 11, 2, 2]
ms-train-0006: SOLVED in 10 moves -> [0, 4, 3, 3, 3, 2, 7, 11, 2, 2]
ms-train-0007: SOLVED in 11 moves -> [0, 4, 3, 3, 3, 3, 7, 11, 2, 2, 2]
ms-train-0008: SOLVED in 6 moves -> [6, 8, 5, 1, 5, 12]
ms-train-0009: SOLVED in 7 moves -> [0, 4, 3, 3, 5, 12, 13]
ms-train-0010: SOLVED in 8 moves -> [6, 8, 

In [2]:
"""
Full AC solver — uses the ac-r2-v1 spec from move_spec.json.
Run from: competition/examples/
"""

import json
from collections import deque
from pathlib import Path

# ============================================================
# STEP 1: LOCATE FILES
# ============================================================
EXAMPLES_DIR = Path(".")
AC_JSONL = EXAMPLES_DIR / "ac.jsonl"
assert AC_JSONL.exists(), "ac.jsonl not found. Run from competition/examples/"

# ============================================================
# STEP 2: PARSER
# ============================================================
def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    generators = gens_part.strip().split()
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        relators.append(encode(chunk.replace("= 1", "").strip()))
    return generators, relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            gens, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "generators": gens,
                "relators": rels,
            })
    return problems

# ============================================================
# STEP 3: FREE REDUCTION AND INVERSION
# ============================================================
def is_inv(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    stack = []
    for ch in word:
        if stack and is_inv(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    return "".join(ch.upper() if ch.islower() else ch.lower()
                   for ch in reversed(word))

# ============================================================
# STEP 4: apply_move — matches move_spec.json (ac-r2-v1)
# ============================================================
def apply_move(rels, m):
    r = list(rels)
    if   m == 0:  r[0] = free_reduce(invert_word(r[0]))
    elif m == 1:  r[1] = free_reduce(invert_word(r[1]))
    elif m == 2:  r[0] = free_reduce(r[0] + r[1])
    elif m == 3:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif m == 4:  r[1] = free_reduce(r[1] + r[0])
    elif m == 5:  r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif m == 6:  r[0] = free_reduce("x" + r[0] + "X")
    elif m == 7:  r[0] = free_reduce("X" + r[0] + "x")
    elif m == 8:  r[0] = free_reduce("y" + r[0] + "Y")
    elif m == 9:  r[0] = free_reduce("Y" + r[0] + "y")
    elif m == 10: r[1] = free_reduce("x" + r[1] + "X")
    elif m == 11: r[1] = free_reduce("X" + r[1] + "x")
    elif m == 12: r[1] = free_reduce("y" + r[1] + "Y")
    elif m == 13: r[1] = free_reduce("Y" + r[1] + "y")
    return r

# ============================================================
# STEP 5: STATE KEY
# ============================================================
def state_key(rels):
    return tuple(sorted(rels))

# ============================================================
# STEP 6: BIDIRECTIONAL BFS
# ============================================================
GOAL = ["x", "y"]

def bidirectional_solve(start_rels, max_depth=14):
    start_key = state_key(start_rels)
    goal_key = state_key(GOAL)
    if start_key == goal_key:
        return []

    fwd = deque([(list(start_rels), [])])
    bwd = deque([(list(GOAL), [])])
    fwd_seen = {start_key: []}
    bwd_seen = {goal_key: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in bwd_seen:
                    return path + [m] + bwd_seen[nk]
                if nk not in fwd_seen:
                    fwd_seen[nk] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in fwd_seen:
                    return fwd_seen[nk] + [m] + path
                if nk not in bwd_seen:
                    bwd_seen[nk] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ============================================================
# STEP 7: SANITY CHECK
# ============================================================
print("=== Sanity check on ms-train-0001 ===")
start = ["XyxyY", "xy"]
known = [0, 9, 1, 2, 9, 0, 4, 1]
state = list(start)
for m in known:
    state = apply_move(state, m)
print("after known moves:", state)
print("goal:", ["x", "y"])
print("MATCH:", sorted(state) == sorted(["x", "y"]))

# ============================================================
# STEP 8: SOLVE FIRST 10 AND WRITE SUBMISSION
# ============================================================
print("\n=== Solving first 10 training problems ===")
problems = load_jsonl(AC_JSONL)[:10]
results = {}
for prob in problems:
    cid = prob["challenge_id"]
    moves = bidirectional_solve(prob["relators"], max_depth=14)
    if moves is not None:
        print(f"{cid}: SOLVED in {len(moves)} moves -> {moves}")
        results[cid] = moves
    else:
        print(f"{cid}: NOT SOLVED")

with open("practice_submission_4.txt", "w") as f:
    for cid, moves in results.items():
        f.write(f"{cid}: [{', '.join(str(m) for m in moves)}]\n")

print(f"\nWrote {len(results)} solutions to practice_submission_4.txt")

=== Sanity check on ms-train-0001 ===
after known moves: ['xyy', 'Y']
goal: ['x', 'y']
MATCH: False

=== Solving first 10 training problems ===
ms-train-0001: SOLVED in 5 moves -> [6, 2, 9, 4, 10]
ms-train-0002: SOLVED in 6 moves -> [6, 2, 11, 8, 4, 8]
ms-train-0003: SOLVED in 7 moves -> [6, 2, 9, 9, 4, 8, 10]
ms-train-0004: SOLVED in 8 moves -> [6, 2, 9, 11, 8, 4, 8, 8]
ms-train-0005: SOLVED in 9 moves -> [6, 2, 9, 9, 9, 4, 8, 8, 10]
ms-train-0006: SOLVED in 10 moves -> [6, 2, 9, 9, 11, 8, 4, 8, 8, 8]
ms-train-0007: SOLVED in 11 moves -> [6, 2, 9, 9, 9, 9, 4, 8, 8, 8, 10]
ms-train-0008: SOLVED in 6 moves -> [9, 3, 8, 1, 0, 4]
ms-train-0009: SOLVED in 7 moves -> [0, 6, 3, 7, 3, 5, 12]
ms-train-0010: SOLVED in 8 moves -> [6, 2, 9, 9, 12, 3, 5, 1]

Wrote 10 solutions to practice_submission_4.txt


In [3]:
"""
Full AC solver — uses the ac-r2-v1 spec from move_spec.json.
Run from: competition/examples/
"""

import json
from collections import deque
from pathlib import Path

EXAMPLES_DIR = Path(".")
AC_JSONL = EXAMPLES_DIR / "ac.jsonl"
assert AC_JSONL.exists(), "ac.jsonl not found."

# ---------- Parser ----------
def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if chunk:
            relators.append(encode(chunk.replace("= 1", "").strip()))
    return gens_part.strip().split(), relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            _, rels = parse_description(obj["description"])
            problems.append({"challenge_id": obj["challenge_id"], "relators": rels})
    return problems

# ---------- Free reduction ----------
def is_inv(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    stack = []
    for ch in word:
        if stack and is_inv(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    return "".join(ch.upper() if ch.islower() else ch.lower()
                   for ch in reversed(word))

# ---------- apply_move: exactly matches ac-r2-v1 spec ----------
def apply_move(rels, m):
    r = list(rels)
    if   m == 0:  r[0] = free_reduce(invert_word(r[0]))
    elif m == 1:  r[1] = free_reduce(invert_word(r[1]))
    elif m == 2:  r[0] = free_reduce(r[0] + r[1])
    elif m == 3:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif m == 4:  r[1] = free_reduce(r[1] + r[0])
    elif m == 5:  r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif m == 6:  r[0] = free_reduce("x" + r[0] + "X")
    elif m == 7:  r[0] = free_reduce("X" + r[0] + "x")
    elif m == 8:  r[0] = free_reduce("y" + r[0] + "Y")
    elif m == 9:  r[0] = free_reduce("Y" + r[0] + "y")
    elif m == 10: r[1] = free_reduce("x" + r[1] + "X")
    elif m == 11: r[1] = free_reduce("X" + r[1] + "x")
    elif m == 12: r[1] = free_reduce("y" + r[1] + "Y")
    elif m == 13: r[1] = free_reduce("Y" + r[1] + "y")
    return r

# ---------- State key: DO NOT sort ----------
def state_key(rels):
    return tuple(rels)

# ---------- Goal: EXACT ordered pair ----------
GOAL = ("x", "y")

# ---------- Bidirectional BFS ----------
def bidirectional_solve(start_rels, max_depth=14):
    start_key = state_key(start_rels)
    if start_key == GOAL:
        return []

    fwd = deque([(list(start_rels), [])])
    bwd = deque([(list(GOAL), [])])
    fwd_seen = {start_key: []}
    bwd_seen = {GOAL: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in bwd_seen:
                    return path + [m] + bwd_seen[nk]
                if nk not in fwd_seen:
                    fwd_seen[nk] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in fwd_seen:
                    return fwd_seen[nk] + [m] + path
                if nk not in bwd_seen:
                    bwd_seen[nk] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ---------- Sanity check ----------
print("=== Sanity check on ms-train-0001 ===")
start = ["XyxyY", "xy"]
known = [0, 9, 1, 2, 9, 0, 4, 1]
state = list(start)
for m in known:
    state = apply_move(state, m)
print("after known moves:", state)
print("goal:", list(GOAL))
print("MATCH:", tuple(state) == GOAL)

# ---------- Solve first 10 and write submission ----------
print("\n=== Solving first 10 training problems ===")
problems = load_jsonl(AC_JSONL)[:10]
results = {}
for prob in problems:
    cid = prob["challenge_id"]
    moves = bidirectional_solve(prob["relators"], max_depth=14)
    if moves is not None:
        print(f"{cid}: SOLVED in {len(moves)} moves -> {moves}")
        results[cid] = moves
    else:
        print(f"{cid}: NOT SOLVED")

with open("practice_submission_5.txt", "w") as f:
    for cid, moves in results.items():
        f.write(f"{cid}: [{', '.join(str(m) for m in moves)}]\n")

print(f"\nWrote {len(results)} solutions to practice_submission_5.txt")

=== Sanity check on ms-train-0001 ===
after known moves: ['xyy', 'Y']
goal: ['x', 'y']
MATCH: False

=== Solving first 10 training problems ===
ms-train-0001: SOLVED in 5 moves -> [6, 2, 9, 4, 10]
ms-train-0002: SOLVED in 6 moves -> [6, 2, 11, 8, 4, 8]
ms-train-0003: SOLVED in 7 moves -> [6, 2, 9, 9, 4, 8, 10]
ms-train-0004: SOLVED in 8 moves -> [6, 2, 9, 11, 8, 4, 8, 8]
ms-train-0005: SOLVED in 9 moves -> [6, 2, 9, 9, 9, 4, 8, 8, 10]
ms-train-0006: SOLVED in 10 moves -> [6, 2, 9, 9, 11, 8, 4, 8, 8, 8]
ms-train-0007: SOLVED in 11 moves -> [6, 2, 9, 9, 9, 9, 4, 8, 8, 8, 10]
ms-train-0008: SOLVED in 6 moves -> [9, 3, 8, 1, 0, 4]
ms-train-0009: SOLVED in 8 moves -> [9, 9, 3, 8, 9, 0, 4, 1]
ms-train-0010: SOLVED in 9 moves -> [6, 2, 9, 9, 9, 3, 3, 4, 13]

Wrote 10 solutions to practice_submission_5.txt


In [4]:
"""
Diagnostic: trace the known valid solution for ms-train-0001 and compare
against the verifier's expected behavior.

Known valid solution (from training_424.json):
  ms-train-0001: [0, 9, 1, 2, 9, 0, 4, 1]
Start relators: XyxyY, xy
Goal relators:  x, y   (exact ordered pair)
"""

# ============================================================
# Encoding: x -> "x", x^-1 -> "X", y -> "y", y^-1 -> "Y"
# ============================================================

def is_inv(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    stack = []
    for ch in word:
        if stack and is_inv(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    return "".join(ch.upper() if ch.islower() else ch.lower()
                   for ch in reversed(word))

# ============================================================
# Move spec (from move_spec.json, ac-r2-v1):
#  0: r0 <- r0^-1
#  1: r1 <- r1^-1
#  2: r0 <- r0 r1
#  3: r0 <- r0 r1^-1
#  4: r1 <- r1 r0
#  5: r1 <- r1 r0^-1
#  6: r0 <- x r0 x^-1
#  7: r0 <- x^-1 r0 x
#  8: r0 <- y r0 y^-1
#  9: r0 <- y^-1 r0 y
# 10: r1 <- x r1 x^-1
# 11: r1 <- x^-1 r1 x
# 12: r1 <- y r1 y^-1
# 13: r1 <- y^-1 r1 y
# ============================================================

def apply_move(rels, m):
    r = list(rels)
    if   m == 0:  r[0] = free_reduce(invert_word(r[0]))
    elif m == 1:  r[1] = free_reduce(invert_word(r[1]))
    elif m == 2:  r[0] = free_reduce(r[0] + r[1])
    elif m == 3:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif m == 4:  r[1] = free_reduce(r[1] + r[0])
    elif m == 5:  r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif m == 6:  r[0] = free_reduce("x" + r[0] + "X")
    elif m == 7:  r[0] = free_reduce("X" + r[0] + "x")
    elif m == 8:  r[0] = free_reduce("y" + r[0] + "Y")
    elif m == 9:  r[0] = free_reduce("Y" + r[0] + "y")
    elif m == 10: r[1] = free_reduce("x" + r[1] + "X")
    elif m == 11: r[1] = free_reduce("X" + r[1] + "x")
    elif m == 12: r[1] = free_reduce("y" + r[1] + "Y")
    elif m == 13: r[1] = free_reduce("Y" + r[1] + "y")
    return r

# ============================================================
# Trace the known solution step by step
# ============================================================
print("=" * 60)
print("Tracing ms-train-0001 with known valid solution")
print("=" * 60)

state = ["XyxyY", "xy"]
known = [0, 9, 1, 2, 9, 0, 4, 1]

print(f"\nstart: {state}")
for i, m in enumerate(known, 1):
    state = apply_move(state, m)
    print(f"after move {i} (id={m}): {state}")

print(f"\nexpected final: ['x', 'y']")
print(f"actual final:   {state}")
print(f"MATCH: {state == ['x', 'y']}")

# ============================================================
# Also test: what if my free_reduce is wrong?
# Show step-by-step internal computation for a couple of moves
# ============================================================
print("\n" + "=" * 60)
print("Detailed trace: what happens when we apply the conjugations")
print("=" * 60)

def explain_conj(word, conjugator, conj_inv, label):
    print(f"\n{label}")
    print(f"  input word:   {word}")
    print(f"  conjugator:   {conjugator}")
    print(f"  conjugated:   {conjugator + word + conj_inv}")
    print(f"  reduced:      {free_reduce(conjugator + word + conj_inv)}")

explain_conj("XyxyY", "Y", "y", "Move 9: r0 <- y^-1 r0 y")
explain_conj("xy", "Y", "y", "Move 9 applied to r1 (should not change r1)")

# ============================================================
# Also test: is the issue maybe with how inverses are encoded?
# ============================================================
print("\n" + "=" * 60)
print("Inversion check")
print("=" * 60)

for w in ["XyxyY", "xy", "xyy", "Y"]:
    print(f"  invert({w}) = {invert_word(w)}  (reduced: {free_reduce(invert_word(w))})")

# ============================================================
# All 14 moves applied to the start state, so we can see what each does
# ============================================================
print("\n" + "=" * 60)
print("All 14 moves applied to start state ['XyxyY', 'xy']")
print("=" * 60)

start = ["XyxyY", "xy"]
for m in range(14):
    result = apply_move(start, m)
    print(f"  move {m:2d}: {result}")

Tracing ms-train-0001 with known valid solution

start: ['XyxyY', 'xy']
after move 1 (id=0): ['XYx', 'xy']
after move 2 (id=9): ['YXYxy', 'xy']
after move 3 (id=1): ['YXYxy', 'YX']
after move 4 (id=2): ['YXY', 'YX']
after move 5 (id=9): ['YYX', 'YX']
after move 6 (id=0): ['xyy', 'YX']
after move 7 (id=4): ['xyy', 'y']
after move 8 (id=1): ['xyy', 'Y']

expected final: ['x', 'y']
actual final:   ['xyy', 'Y']
MATCH: False

Detailed trace: what happens when we apply the conjugations

Move 9: r0 <- y^-1 r0 y
  input word:   XyxyY
  conjugator:   Y
  conjugated:   YXyxyYy
  reduced:      YXyxy

Move 9 applied to r1 (should not change r1)
  input word:   xy
  conjugator:   Y
  conjugated:   Yxyy
  reduced:      Yxyy

Inversion check
  invert(XyxyY) = yYXYx  (reduced: XYx)
  invert(xy) = YX  (reduced: YX)
  invert(xyy) = YYX  (reduced: YYX)
  invert(Y) = y  (reduced: y)

All 14 moves applied to start state ['XyxyY', 'xy']
  move  0: ['XYx', 'xy']
  move  1: ['XyxyY', 'YX']
  move  2: ['Xyxxy'

In [6]:
"""
Parallel solver for all 424 training problems.
Run from: competition/examples/
"""

import json
from collections import deque
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

# ============================================================
# ENCODING AND REDUCTION
# ============================================================
def is_inv(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    stack = []
    for ch in word:
        if stack and is_inv(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    return "".join(ch.upper() if ch.islower() else ch.lower()
                   for ch in reversed(word))

def apply_move(rels, m):
    r = list(rels)
    if   m == 0:  r[0] = free_reduce(invert_word(r[0]))
    elif m == 1:  r[1] = free_reduce(invert_word(r[1]))
    elif m == 2:  r[0] = free_reduce(r[0] + r[1])
    elif m == 3:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif m == 4:  r[1] = free_reduce(r[1] + r[0])
    elif m == 5:  r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif m == 6:  r[0] = free_reduce("x" + r[0] + "X")
    elif m == 7:  r[0] = free_reduce("X" + r[0] + "x")
    elif m == 8:  r[0] = free_reduce("y" + r[0] + "Y")
    elif m == 9:  r[0] = free_reduce("Y" + r[0] + "y")
    elif m == 10: r[1] = free_reduce("x" + r[1] + "X")
    elif m == 11: r[1] = free_reduce("X" + r[1] + "x")
    elif m == 12: r[1] = free_reduce("y" + r[1] + "Y")
    elif m == 13: r[1] = free_reduce("Y" + r[1] + "y")
    return r

GOAL = ("x", "y")

def state_key(rels):
    return tuple(rels)

def bidirectional_solve(start_rels, max_depth=20):
    start_key = state_key(start_rels)
    if start_key == GOAL:
        return []

    fwd = deque([(list(start_rels), [])])
    bwd = deque([(list(GOAL), [])])
    fwd_seen = {start_key: []}
    bwd_seen = {GOAL: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in bwd_seen:
                    return path + [m] + bwd_seen[nk]
                if nk not in fwd_seen:
                    fwd_seen[nk] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in fwd_seen:
                    return fwd_seen[nk] + [m] + path
                if nk not in bwd_seen:
                    bwd_seen[nk] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ============================================================
# PARSER
# ============================================================
def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if chunk:
            relators.append(encode(chunk.replace("= 1", "").strip()))
    return gens_part.strip().split(), relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            _, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "relators": rels,
            })
    return problems

# ============================================================
# WORKER: solve one problem
# ============================================================
def solve_one(args):
    cid, rels = args
    moves = bidirectional_solve(rels, max_depth=20)
    return cid, moves

# ============================================================
# MAIN: parallel solve all problems
# ============================================================
if __name__ == "__main__":
    AC_JSONL = Path(".") / "ac.jsonl"
    problems = load_jsonl(AC_JSONL)
    print(f"Loaded {len(problems)} problems.")

    tasks = [(p["challenge_id"], p["relators"]) for p in problems]
    results = {}

    import os
    n_workers = min(os.cpu_count() or 4, 8)
    print(f"Using {n_workers} workers.\n")

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(solve_one, t): t[0] for t in tasks}
        done = 0
        for fut in as_completed(futures):
            cid, moves = fut.result()
            done += 1
            if moves is not None:
                results[cid] = moves
                print(f"[{done}/{len(tasks)}] {cid}: SOLVED ({len(moves)} moves)")
            else:
                print(f"[{done}/{len(tasks)}] {cid}: NOT SOLVED")

    with open("practice_submission_parallel.txt", "w") as f:
        for cid, moves in results.items():
            f.write(f"{cid}: [{', '.join(str(m) for m in moves)}]\n")

    print(f"\nWrote {len(results)}/{len(problems)} solutions to practice_submission_parallel.txt")

Loaded 424 problems.
Using 4 workers.



BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [5]:
"""
Full AC solver with diagnostics.
Run from: competition/examples/
"""

import json
from collections import deque
from pathlib import Path

# ============================================================
# ENCODING:  x -> "x",  x^-1 -> "X",  y -> "y",  y^-1 -> "Y"
# ============================================================

def is_inv(a, b):
    return a.lower() == b.lower() and a != b

def free_reduce(word):
    """Single left-fold stack cancellation."""
    stack = []
    for ch in word:
        if stack and is_inv(stack[-1], ch):
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)

def invert_word(word):
    """Reverse and swap case: 'xY' -> 'yX'."""
    return "".join(ch.upper() if ch.islower() else ch.lower()
                   for ch in reversed(word))

# ============================================================
# MOVE TABLE (ac-r2-v1)
# ============================================================
def apply_move(rels, m):
    r = list(rels)
    if   m == 0:  r[0] = free_reduce(invert_word(r[0]))
    elif m == 1:  r[1] = free_reduce(invert_word(r[1]))
    elif m == 2:  r[0] = free_reduce(r[0] + r[1])
    elif m == 3:  r[0] = free_reduce(r[0] + invert_word(r[1]))
    elif m == 4:  r[1] = free_reduce(r[1] + r[0])
    elif m == 5:  r[1] = free_reduce(r[1] + invert_word(r[0]))
    elif m == 6:  r[0] = free_reduce("x" + r[0] + "X")
    elif m == 7:  r[0] = free_reduce("X" + r[0] + "x")
    elif m == 8:  r[0] = free_reduce("y" + r[0] + "Y")
    elif m == 9:  r[0] = free_reduce("Y" + r[0] + "y")
    elif m == 10: r[1] = free_reduce("x" + r[1] + "X")
    elif m == 11: r[1] = free_reduce("X" + r[1] + "x")
    elif m == 12: r[1] = free_reduce("y" + r[1] + "Y")
    elif m == 13: r[1] = free_reduce("Y" + r[1] + "y")
    return r

# ============================================================
# PARSER
# ============================================================
def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if chunk:
            relators.append(encode(chunk.replace("= 1", "").strip()))
    return gens_part.strip().split(), relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            _, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "relators": rels,
            })
    return problems

# ============================================================
# BIDIRECTIONAL BFS
# ============================================================
GOAL = ("x", "y")

def state_key(rels):
    return tuple(rels)

def bidirectional_solve(start_rels, max_depth=20):
    start_key = state_key(start_rels)
    if start_key == GOAL:
        return []

    fwd = deque([(list(start_rels), [])])
    bwd = deque([(list(GOAL), [])])
    fwd_seen = {start_key: []}
    bwd_seen = {GOAL: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in bwd_seen:
                    return path + [m] + bwd_seen[nk]
                if nk not in fwd_seen:
                    fwd_seen[nk] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                nk = state_key(ns)
                if nk in fwd_seen:
                    return fwd_seen[nk] + [m] + path
                if nk not in bwd_seen:
                    bwd_seen[nk] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ============================================================
# DIAGNOSTIC: trace known solution for ms-train-0001
# ============================================================
print("=" * 60)
print("DIAGNOSTIC: ms-train-0001, known solution [0,9,1,2,9,0,4,1]")
print("=" * 60)
state = ["XyxyY", "xy"]
print(f"  start:  {state}")
for i, m in enumerate([0, 9, 1, 2, 9, 0, 4, 1], 1):
    state = apply_move(state, m)
    print(f"  after move {i} (id={m}): {state}")
print(f"  goal:   ['x', 'y']")
print(f"  MATCH:  {state == ['x', 'y']}")

# ============================================================
# DIAGNOSTIC: apply all 14 moves to start state
# ============================================================
print("\n" + "=" * 60)
print("DIAGNOSTIC: all 14 moves applied to start ['XyxyY', 'xy']")
print("=" * 60)
for m in range(14):
    print(f"  move {m:2d}: {apply_move(['XyxyY', 'xy'], m)}")

# ============================================================
# SOLVE ALL TRAINING PROBLEMS
# ============================================================
EXAMPLES_DIR = Path(".")
AC_JSONL = EXAMPLES_DIR / "ac.jsonl"

print("\n" + "=" * 60)
print("SOLVING ALL TRAINING PROBLEMS")
print("=" * 60)

problems = load_jsonl(AC_JSONL)
print(f"Loaded {len(problems)} problems.\n")

results = {}
for prob in problems:
    cid = prob["challenge_id"]
    moves = bidirectional_solve(prob["relators"], max_depth=20)
    if moves is not None:
        results[cid] = moves
        print(f"{cid}: SOLVED in {len(moves)} moves")
    else:
        print(f"{cid}: NOT SOLVED within depth 20")

with open("practice_submission_full.txt", "w") as f:
    for cid, moves in results.items():
        f.write(f"{cid}: [{', '.join(str(m) for m in moves)}]\n")

print(f"\nWrote {len(results)}/{len(problems)} solutions to practice_submission_full.txt")

DIAGNOSTIC: ms-train-0001, known solution [0,9,1,2,9,0,4,1]
  start:  ['XyxyY', 'xy']
  after move 1 (id=0): ['XYx', 'xy']
  after move 2 (id=9): ['YXYxy', 'xy']
  after move 3 (id=1): ['YXYxy', 'YX']
  after move 4 (id=2): ['YXY', 'YX']
  after move 5 (id=9): ['YYX', 'YX']
  after move 6 (id=0): ['xyy', 'YX']
  after move 7 (id=4): ['xyy', 'y']
  after move 8 (id=1): ['xyy', 'Y']
  goal:   ['x', 'y']
  MATCH:  False

DIAGNOSTIC: all 14 moves applied to start ['XyxyY', 'xy']
  move  0: ['XYx', 'xy']
  move  1: ['XyxyY', 'YX']
  move  2: ['Xyxxy', 'xy']
  move  3: ['XyxYX', 'xy']
  move  4: ['XyxyY', 'xyXyx']
  move  5: ['XyxyY', 'xyXYx']
  move  6: ['y', 'xy']
  move  7: ['XXyxx', 'xy']
  move  8: ['yXyxY', 'xy']
  move  9: ['YXyxy', 'xy']
  move 10: ['XyxyY', 'xxyX']
  move 11: ['XyxyY', 'yx']
  move 12: ['XyxyY', 'yx']
  move 13: ['XyxyY', 'Yxyy']

SOLVING ALL TRAINING PROBLEMS
Loaded 424 problems.

ms-train-0001: SOLVED in 5 moves
ms-train-0002: SOLVED in 6 moves
ms-train-0003: SOLV

KeyboardInterrupt: 

In [ ]:
"""
Full AC solver matching the verifier's core.py exactly.
"""

import json
from collections import deque
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------- Core semantics (from verifier/core.py) ----------
def free_reduce(word):
    out = []
    for a in word:
        if out and out[-1] == -a:
            out.pop()
        else:
            out.append(a)
    return tuple(out)

def invert(word):
    return tuple(-a for a in reversed(word))

GENS = (1, -1, 2, -2)

def apply_move(state, m):
    r0, r1 = state
    if m == 0:  return (invert(r0), r1)
    if m == 1:  return (r0, invert(r1))
    if m == 2:  return (free_reduce(r0 + r1), r1)
    if m == 3:  return (free_reduce(r0 + invert(r1)), r1)
    if m == 4:  return (r0, free_reduce(r1 + r0))
    if m == 5:  return (r0, free_reduce(r1 + invert(r0)))
    if 6 <= m <= 9:
        g = GENS[m - 6]
        return (free_reduce((g,) + r0 + (-g,)), r1)
    if 10 <= m <= 13:
        g = GENS[m - 10]
        return (r0, free_reduce((g,) + r1 + (-g,)))
    raise ValueError("bad move id: %r" % (m,))

# ---------- Parser: strings -> integers ----------
CHAR_TO_INT = {"x": 1, "X": -1, "y": 2, "Y": -2}

def encode(word):
    return (word.replace("x^-1", "X")
                .replace("y^-1", "Y")
                .replace(" ", ""))

def parse_description(desc):
    s = desc.strip().replace("Presentation:", "").strip().rstrip(".").strip()
    gens_part, rels_part = s.strip("<>").split("|", 1)
    relators = []
    for chunk in rels_part.split(";"):
        chunk = chunk.strip()
        if chunk:
            w = encode(chunk.replace("= 1", "").strip())
            relators.append(tuple(CHAR_TO_INT[c] for c in w))
    return gens_part.strip().split(), relators

def load_jsonl(path):
    problems = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            _, rels = parse_description(obj["description"])
            problems.append({
                "challenge_id": obj["challenge_id"],
                "initial": (rels[0], rels[1]),
            })
    return problems

# ---------- Goal ----------
TARGET = ((1,), (2,))     # exact ordered pair (x, y)

def state_key(state):
    return state

# ---------- Bidirectional BFS ----------
def bidirectional_solve(initial, max_depth=20):
    if initial == TARGET:
        return []

    fwd = deque([(initial, [])])
    bwd = deque([(TARGET, [])])
    fwd_seen = {initial: []}
    bwd_seen = {TARGET: []}

    depth = 0
    while (fwd or bwd) and depth <= max_depth:
        for _ in range(len(fwd)):
            state, path = fwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                if ns in bwd_seen:
                    return path + [m] + bwd_seen[ns]
                if ns not in fwd_seen:
                    fwd_seen[ns] = path + [m]
                    fwd.append((ns, path + [m]))
        for _ in range(len(bwd)):
            state, path = bwd.popleft()
            for m in range(14):
                ns = apply_move(state, m)
                if ns in fwd_seen:
                    return fwd_seen[ns] + [m] + path
                if ns not in bwd_seen:
                    bwd_seen[ns] = path + [m]
                    bwd.append((ns, path + [m]))
        depth += 1
    return None

# ---------- Sanity check: known solution ----------
print("=== Sanity check on ms-train-0001 ===")
state = ((-1, 2, 1, 2, -2), (1, 2))
known = [0, 9, 1, 2, 9, 0, 4, 1]
for m in known:
    state = apply_move(state, m)
print(f"after known moves: {state}")
print(f"target: {TARGET}")
print(f"MATCH: {state == TARGET}")

# ---------- Solve all 424 training problems ----------
def solve_one(args):
    cid, initial = args
    return cid, bidirectional_solve(initial, max_depth=20)

if __name__ == "__main__":
    AC_JSONL = Path(".") / "ac.jsonl"
    problems = load_jsonl(AC_JSONL)
    print(f"\nLoaded {len(problems)} problems.")

    tasks = [(p["challenge_id"], p["initial"]) for p in problems]
    results = {}

    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(solve_one, t) for t in tasks]
        done = 0
        for fut in as_completed(futures):
            cid, moves = fut.result()
            done += 1
            if moves is not None:
                results[cid] = moves
                print(f"[{done}/{len(tasks)}] {cid}: SOLVED ({len(moves)})")
            else:
                print(f"[{done}/{len(tasks)}] {cid}: NOT SOLVED")

    with open("practice_submission_int.txt", "w") as f:
        for cid, moves in results.items():
            f.write(f"{cid}: [{', '.join(str(m) for m in moves)}]\n")

    print(f"\nWrote {len(results)}/{len(problems)} solutions.")